In [1]:
import pandas as pd
import numpy as np
import re
import datetime
from collections import Counter

In [2]:
# le fichier sur trouve a cette URL:
#url = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe-v2-tertiaire-2/lines?size=10000&format=csv&after=10000%2C965634&header=true"

# on le charge directement dans une dataframe Pandas
#data = pd.read_csv(url)

In [3]:
data = pd.read_csv("data/dpe-v2-tertiaire-2.csv")
print(data.shape)

(359761, 63)


# Data exploration

In [4]:
data.head()

,N°DPE,Date_réception_DPE,Date_établissement_DPE,Date_visite_diagnostiqueur,Modèle_DPE,N°_DPE_remplacé,Date_fin_validité_DPE,Version_DPE,N°_DPE_immeuble_associé,Méthode_du_DPE,...,Type_énergie_n°2,Type_usage_énergie_n°2,Frais_annuel_énergie_n°2,Année_relève_conso_énergie_n°2,Conso_é_finale_énergie_n°3,Conso_é_primaire_énergie_n°3,Type_énergie_n°3,Type_usage_énergie_n°3,Frais_annuel_énergie_n°3,Année_relève_conso_énergie_n°3
0,2363T1680837R,2023-05-23,2023-05-22,2023-05-11,DPE 2006 tertiaire et ERP,NaN,2033-05-21,2.2,NaN,dpe tertiaire vierge,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2269T2615953W,2022-11-05,2022-11-04,2022-10-19,DPE 2006 tertiaire et ERP,NaN,2032-11-03,2.2,NaN,dpe tertiaire facture,...,Électricité,périmètre de l'usage inconnu,1167.0,2021.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2357T4209637Z,2023-12-07,2023-12-06,2023-12-06,DPE 2006 tertiaire et ERP,NaN,2033-12-05,2.3,NaN,dpe tertiaire vierge dans un bâtiment de logement,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2404T3934296Y,2024-11-07,2024-11-06,2024-07-29,DPE 2006 tertiaire et ERP,2404T3934235P,2034-11-05,2.4,NaN,dpe tertiaire facture,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2360T0127880L,2023-01-16,2023-01-15,2023-01-05,DPE 2006 tertiaire et ERP,NaN,2033-01-14,2.2,NaN,dpe tertiaire vierge,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
data.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 359761 entries, 0 to 359760
Data columns (total 63 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   N°DPE                              359761 non-null  object 
 1   Date_réception_DPE                 359761 non-null  object 
 2   Date_établissement_DPE             359761 non-null  object 
 3   Date_visite_diagnostiqueur         359761 non-null  object 
 4   Modèle_DPE                         359761 non-null  object 
 5   N°_DPE_remplacé                    26144 non-null   object 
 6   Date_fin_validité_DPE              359761 non-null  object 
 7   Version_DPE                        359761 non-null  float64
 8   N°_DPE_immeuble_associé            1036 non-null    object 
 9   Méthode_du_DPE                     352262 non-null  object 
 10  N°_immatriculation_copropriété     1405 non-null    object 
 11  Invariant_fiscal_logement          0 no

# Data preprocessing

## Rename columns

In [6]:
def rename_columns(columns):
    # en minuscule
    columns = [col.lower() for col in columns]
    # regex de remplacement
    rgxs = [
        (r"[°|/|']", "_"),
        (r"²", "2"),
        (r"[(|)]", ""),
        (r"é|è", "e"),
        (r"_+", "_"),
    ]
    # on remplace toutes les colonnes une par une
    for rgx in rgxs:
        columns = [re.sub(rgx[0], rgx[1], col) for col in columns]

    return columns

In [7]:
data.columns = rename_columns(data.columns)
data.columns

Index(['n_dpe', 'date_reception_dpe', 'date_etablissement_dpe',
       'date_visite_diagnostiqueur', 'modele_dpe', 'n_dpe_remplace',
       'date_fin_validite_dpe', 'version_dpe', 'n_dpe_immeuble_associe',
       'methode_du_dpe', 'n_immatriculation_copropriete',
       'invariant_fiscal_logement', 'etiquette_dpe', 'etiquette_ges',
       'conso_kwhep_m2_an', 'emission_ges_kgco2_m2_an', 'annee_construction',
       'categorie_erp', 'periode_construction', 'secteur_activite',
       'nombre_occupant', 'surface_shon', 'surface_utile',
       'type_energie_principale_chauffage', 'adresse_brute', 'nom_commune_ban',
       'code_insee_ban', 'n_voie_ban', 'identifiant_ban', 'adresse_ban',
       'code_postal_ban', 'score_ban', 'nom_rue_ban',
       'coordonnee_cartographique_x_ban', 'coordonnee_cartographique_y_ban',
       'code_postal_brut', 'n_etage_appartement', 'nom_residence',
       'complement_d_adresse_bâtiment', 'cage_d_escalier',
       'complement_d_adresse_logement', 'statut_geo

## Drop columns

In [8]:
columns_int = [
    "version_dpe",
    "surface_utile",
    "conso_kwhep_m2_an",
    "conso_e_finale_energie_n_1",
]

columns_categorical = [
    "periode_construction",
    "secteur_activite",
    "type_energie_principale_chauffage",
    "type_energie_n_1",
    "type_usage_energie_n_1",
]

target = 'etiquette_dpe'
id_col = 'n_dpe'

train_columns = columns_int + columns_categorical

In [9]:
train_columns

['version_dpe',
 'surface_utile',
 'conso_kwhep_m2_an',
 'conso_e_finale_energie_n_1',
 'periode_construction',
 'secteur_activite',
 'type_energie_principale_chauffage',
 'type_energie_n_1',
 'type_usage_energie_n_1']

In [10]:
for col in columns_int:
    data[col] = data[col].fillna(-1.0)
    data.loc[data[col] == "", col] = -1.0
    data[col] = data[col].astype(int)

In [11]:
data[columns_int].describe(percentiles = [0.99])

,version_dpe,surface_utile,conso_kwhep_m2_an,conso_e_finale_energie_n_1
count,359761.000000,359761.000000,3.597610e+05,3.597610e+05
mean,1.922551,422.203949,2.963608e+02,7.739098e+04
std,0.267302,2805.467982,1.379814e+04,6.679152e+06
min,1.000000,0.000000,-5.975000e+03,-2.147484e+09
50%,2.000000,100.000000,1.400000e+01,6.290000e+02
99%,2.000000,6522.600000,1.237000e+03,1.054036e+06
max,2.000000,361381.000000,3.019960e+06,2.005735e+09


In [12]:
data = data[data['surface_utile'] < 9800]
data = data[data['conso_kwhep_m2_an'] < 2000]

In [13]:
for col in columns_categorical:
    data[col] = data[col].fillna("")

## Encoding categorical columns

In [14]:
data['type_energie_principale_chauffage'].value_counts()

type_energie_principale_chauffage
                                           322010
Électricité                                 23696
Gaz naturel                                  7404
Réseau de Chauffage urbain                    995
Fioul domestique                              972
autre combustible fossile                     606
Bois – Bûches                                 176
Bois – Granulés (pellets) ou briquettes       151
Propane                                       114
GPL                                            75
Bois – Plaquettes forestières                  18
Bois – Plaquettes d’industrie                  11
Butane                                          9
Charbon                                         1
Name: count, dtype: int64

In [15]:
data['type_usage_energie_n_1'].value_counts(dropna=False)

type_usage_energie_n_1
                                      166538
périmètre de l'usage inconnu          144693
Chauffage                              29408
Eau Chaude sanitaire                    7974
Eclairage                               3176
Ascenseur(s)                            1901
Autres usages                           1056
Refroidissement                          727
Production d'électricité à demeure       319
Bureautique                              227
auxiliaires et ventilation               219
Name: count, dtype: int64

In [16]:
map_type_energie = {
    "non renseigné": -1,
    "Électricité": 1,
    "Électricité d'origine renouvelable utilisée dans le bâtiment": 1,
    "Gaz naturel": 2,
    "Butane": 2,
    "Propane": 2,
    "GPL": 2,
    "Fioul domestique": 3,
    "Réseau de Chauffage urbain": 4,
    "Charbon": 5,
    "autre combustible fossile": 5,
    "Bois - Bûches": 6,
    "Bois - Plaquettes forestières": 6,
    "Bois - Granulés (pellets) ou briquettes": 6,
    "Bois - Plaquettes d'industrie": 6,
}

In [17]:
map_type_usage = {
    "non renseigné": -1,
    "périmètre de l'usage inconnu": -1,
    "Chauffage": 1,
    "Eau Chaude sanitaire": 2,
    "Eclairage": 3,
    "Refroidissement": 4,
    "auxiliaires et ventilation": 4,
    "Ascenseur(s)": 5,
    "Autres usages": 6,
    "Bureautique": 6,
    "Abonnements": 6,
    "Production d'électricité à demeure": 6,
}

In [18]:
map_secteur_activite = {
    "autres tertiaires non ERP": 1,
    "M : Magasins de vente, centres commerciaux": 2,
    "W : Administrations, banques, bureaux": 3,
    "locaux d'entreprise (bureaux)": 4,
    "J : Structures d’accueil pour personnes âgées ou personnes handicapées": 5,
    "N : Restaurants et débits de boisson": 6,
    "U : Établissements de soins": 7,
    "GHW : Bureaux": 8,
    "R : Établissements d’éveil, d’enseignement, de formation, centres de vacances, centres de loisirs sans hébergement": 9,
    "O : Hôtels et pensions de famille": 10,
    "GHZ : Usage mixte": 11,
    "X : Établissements sportifs couverts": 12,
    "L : Salles d'auditions, de conférences, de réunions, de spectacles ou à usage multiple": 13,
    "T : Salles d'exposition à vocation commerciale": 14,
    "P : Salles de danse et salles de jeux": 15,
    "GHR : Enseignement": 16,
    "V : Établissements de divers cultes": 17,
    "S : Bibliothèques, centres de documentation": 18,
    "OA : Hôtels-restaurants d'Altitude": 19,
    "GHU : Usage sanitaire": 20,
    "PA : Établissements de Plein Air": 21,
    "GHA : Habitation": 22,
    "GHO : Hôtel": 23,
    "Y : Musées": 24,
    "PS : Parcs de Stationnement couverts": 25,
    "GHTC : tour de contrôle": 26,
    "REF : REFuges de montagne": 27,
    "GA : Gares Accessibles au public (chemins de fer, téléphériques, remonte-pentes...)": 28,
    "CTS : Chapiteaux, Tentes et Structures toile": 29,
    "GHS : Dépôt d'archives": 30,
}

In [19]:
map_periode_construction = {
    "avant 1948": 0,
    "1948-1974": 1,
    "1975-1977": 2,
    "1978-1982": 3,
    "1983-1988": 4,
    "1989-2000": 5,
    "2001-2005": 6,
    "2006-2012": 7,
    "2013-2021": 8,
    "après 2021": 9,
}

In [20]:
map_target = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

In [21]:
def encode_categorical_with_map(data, column, mapping, default_unknown=""):
    # valeurs possibles
    valid_values = list(mapping.keys())
    # les valeurs inconnues
    data.loc[~data[column].isin(valid_values), column] = default_unknown
    # valeurs manquantes et inconnues
    mapping[default_unknown] = -1
    # encodage des valeurs connues
    data[column] = data[column].apply(lambda d: mapping[d])
    return data[column]

In [22]:
mappings = [
    map_periode_construction,
    map_secteur_activite,
    map_type_energie,
    map_type_usage,
    map_type_usage,
]

for col, mapping in zip(columns_categorical, mappings):
    data[col] = encode_categorical_with_map(data, col, mapping)

In [23]:
data[target] = encode_categorical_with_map(data, target, map_target)

In [24]:
all_columns = [id_col] + train_columns + [target]
data = data[all_columns]

data.reset_index(inplace = True, drop = True)

In [25]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356238 entries, 0 to 356237
Data columns (total 11 columns):
 #   Column                             Non-Null Count   Dtype 
---  ------                             --------------   ----- 
 0   n_dpe                              356238 non-null  object
 1   version_dpe                        356238 non-null  int32 
 2   surface_utile                      356238 non-null  int32 
 3   conso_kwhep_m2_an                  356238 non-null  int32 
 4   conso_e_finale_energie_n_1         356238 non-null  int32 
 5   periode_construction               356238 non-null  int64 
 6   secteur_activite                   356238 non-null  int64 
 7   type_energie_principale_chauffage  356238 non-null  int64 
 8   type_energie_n_1                   356238 non-null  int64 
 9   type_usage_energie_n_1             356238 non-null  int64 
 10  etiquette_dpe                      356238 non-null  int64 
dtypes: int32(4), int64(6), object(1)
memory usage: 24.5+

## Train test split

In [26]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

In [27]:
# # load data
# input_file = output_file
# data = pd.read_csv(input_file)
# # shuffle
# data = data.sample(frac=1, random_state=808).reset_index(drop=True)


In [28]:
data.columns

Index(['n_dpe', 'version_dpe', 'surface_utile', 'conso_kwhep_m2_an',
       'conso_e_finale_energie_n_1', 'periode_construction',
       'secteur_activite', 'type_energie_principale_chauffage',
       'type_energie_n_1', 'type_usage_energie_n_1', 'etiquette_dpe'],
      dtype='object')

In [35]:
# Assuming the last column is the target variable
X = data.iloc[:, :-1]  # Features
y = data.iloc[:, -1]  # Target variable
assert y.name == "etiquette_dpe"

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42
)

In [36]:
X_train = X_train.drop(columns=["n_dpe"])
id_test = X_test["n_dpe"]
X_test = X_test.drop(columns=["n_dpe"])

In [37]:
id_test

5567      2476T1182823V
45592     2245T0946862D
164910    2131T0385452W
304964    2293T0183941W
348440    2301T3166931T
              ...      
171463    2413T2423365O
148086    2356T3437676D
144252    2475T4525770R
293672    2321T1298265T
298240    2395T3029801D
Name: n_dpe, Length: 142496, dtype: object

In [38]:
X_test

,version_dpe,surface_utile,conso_kwhep_m2_an,conso_e_finale_energie_n_1,periode_construction,secteur_activite,type_energie_principale_chauffage,type_energie_n_1,type_usage_energie_n_1
5567,2,100,377,20587,5,2,-1,-1,-1
45592,2,100,417,35215,1,5,-1,-1,5
164910,1,100,0,0,1,8,-1,-1,-1
304964,2,100,352,6700,1,2,-1,-1,-1
348440,2,100,79,2150,1,5,-1,-1,-1
...,...,...,...,...,...,...,...,...,...
171463,2,100,-1,-1,1,5,-1,-1,-1
148086,2,100,157,3765,5,7,-1,-1,-1
144252,2,100,-1,-1,0,2,-1,-1,-1
293672,2,93,125,10056,5,1,2,-1,1


In [39]:
rf = RandomForestClassifier()

# Define the parameter grid
param_grid = {
        "n_estimators": [200, 300],  # Number of trees
        "max_depth": [10],  # Maximum depth of the trees
        "min_samples_leaf": [1, 5],  # Maximum depth of the trees
}

# Setup GridSearchCV with k-fold cross-validation
cv = KFold(n_splits=3, random_state=84, shuffle=True)

grid_search = GridSearchCV(
    estimator=rf, param_grid=param_grid, cv=cv, scoring="accuracy", verbose=1
)

# Fit the model
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


GridSearchCV(cv=KFold(n_splits=3, random_state=84, shuffle=True),
             estimator=RandomForestClassifier(),
             param_grid={'max_depth': [10], 'min_samples_leaf': [1, 5],
                         'n_estimators': [200, 300]},
             scoring='accuracy', verbose=1)

In [40]:
# Best parameters and best score
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_}")
print(f"Best model: {grid_search.best_estimator_}")

# Evaluate on the test set
yhat = grid_search.predict(X_test)
print(classification_report(y_test, yhat))

Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 200}
Best cross-validation score: 0.930654713924611
Best model: RandomForestClassifier(max_depth=10, n_estimators=200)
              precision    recall  f1-score   support

          -1       1.00      1.00      1.00     66558
           1       0.99      0.89      0.94      7100
           2       0.89      0.85      0.87     10290
           3       0.89      0.89      0.89     19595
           4       0.85      0.91      0.88     17225
           5       0.81      0.88      0.84      9672
           6       0.88      0.72      0.79      4745
           7       0.87      0.89      0.88      7311

    accuracy                           0.93    142496
   macro avg       0.90      0.88      0.89    142496
weighted avg       0.93      0.93      0.93    142496



In [41]:
# regroup into predictions dataframe
probabilities = grid_search.predict_proba(X_test)

In [42]:

predictions = pd.DataFrame()
predictions.index = id_test
predictions["prob"] = np.max(probabilities, axis=1)
predictions["yhat"] = yhat
predictions["y"] = y_test.values
print(predictions.head())

                   prob  yhat  y
n_dpe                           
2476T1182823V  0.603882     5  5
2245T0946862D  0.365361     6  6
2131T0385452W  0.999946     1  1
2293T0183941W  0.695998     5  5
2301T3166931T  0.780809     2  2


In [43]:
    # feature importance
    feature_importances = grid_search.best_estimator_.feature_importances_
    feature_names = X_train.columns

    # Create a dictionary mapping feature names to their importance
    importance_dict = dict(zip(feature_names, feature_importances))
    importance_dict = dict(
        sorted(importance_dict.items(), key=lambda item: item[1], reverse=True)
    )

    print(importance_dict)

{'conso_kwhep_m2_an': 0.6513136615027479, 'conso_e_finale_energie_n_1': 0.2697558431431948, 'secteur_activite': 0.020063110311094378, 'type_usage_energie_n_1': 0.016291482773775384, 'version_dpe': 0.01563814841572058, 'type_energie_principale_chauffage': 0.010323537370190844, 'surface_utile': 0.009456791855432927, 'periode_construction': 0.0071574246278431845, 'type_energie_n_1': 0.0}
